# Comparación de algoritmos con espacios de búsqueda comparables (R2-3)

Notebook independiente para responder al Revisor 2 (comentario 3): reproduce la Fase 1 original (muestra de 200,000, semilla 42) y amplía DBSCAN a 2D-5D (antes solo 2D), HDBSCAN a min_cluster_size más realistas (50-1000), y prueba sensibilidad de GMM a covariance_type/n_init.

**Cómo correrlo:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno` → CPU (no necesita GPU).
2. Ajusta la ruta de tu `df_maestra.csv` en la celda de carga de datos si no está en `/content/drive/MyDrive/Proyecto/`.
3. Corre todas las celdas en orden (`Entorno de ejecución` → `Ejecutar todas`).
4. Los resultados se guardan automáticamente en tu Drive, en `/content/drive/MyDrive/Proyecto/comparacion_algoritmos/`, por si la sesión se desconecta.

In [ ]:
!pip install -q umap-learn hdbscan

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
import umap.umap_ as umap_cpu
import warnings
warnings.filterwarnings("ignore")
print("✅ Librerías listas")

## 1. Cargar los datos desde Google Drive

Ajusta la ruta si tu `df_maestra.csv` está en otra carpeta de tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Proyecto/df_maestra.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed|^Column1')]
OUT_DIR = '/content/drive/MyDrive/Proyecto/comparacion_algoritmos'
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Shape original: {df.shape}")
df.head(3)

## 2. Preprocesamiento (idéntico al notebook original)

In [ ]:
def preprocesar_saber_pro(df_raw, sample_n=200_000, random_state=42):
    df_limpio = df_raw.copy()

    mapa_bano = {'1': 1, '2': 2, '3 o 4': 3, '5 o 6': 5, 'MAS DE 6': 6, 'NINGUNA': 0}
    mapa_estrato = {'Sin estrato': 0, 'Estrato 1': 1, 'Estrato 2': 2,
                     'Estrato 3': 3, 'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6}
    mapa_valormatricula = {
        'Sin costo': 0, 'Menos de 500 mil': 1,
        'Entre 500 mil y menos de 1 millón': 2,
        'Entre 1 millón y menos de 2.5 millones': 3,
        'Entre 2.5 millones y menos de 4 millones': 4,
        'Entre 4 millones y menos de 5.5 millones': 5,
        'Entre 5.5 millones y menos de 7 millones': 6,
        'Más de 7 millones': 7}
    mapa_educ = {
        'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
        'Secundaria (Bachillerato) incompleta': 3,
        'Secundaria (Bachillerato) completa': 4,
        'Técnica o tecnológica incompleta': 5,
        'Técnica o tecnológica completa': 6,
        'Educación profesional incompleta': 7,
        'EDUCACIÓN PROFESIONAL COMPLETA': 8, 'POSTGRADO': 9}
    mapeo_horas = {'0': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
                    'Entre 21 y 30 horas': 3, 'Más de 30 horas': 4}
    mapeo_semestre = {str(i).zfill(2): i for i in range(1, 12)}
    mapeo_semestre['12 o más'] = 12

    mapeables = {
        'FAMI_CUANTOSCOMPARTEBAÑO':      mapa_bano,
        'FAMI_ESTRATOVIVIENDA':          mapa_estrato,
        'ESTU_VALORMATRICULAUNIVERSIDAD': mapa_valormatricula,
        'FAMI_EDUCACIONPADRE':           mapa_educ,
        'FAMI_EDUCACIONMADRE':           mapa_educ,
        'ESTU_HORASSEMANATRABAJA':       mapeo_horas,
        'ESTU_SEMESTRECURSA':            mapeo_semestre,
    }
    for col, mapa in mapeables.items():
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].map(mapa)

    columnas_puntaje = [
        'MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
        'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']
    columnas_ordinales = [
        'FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
        'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']
    columnas_nominales = [
        'ESTU_TITULOOBTENIDOBACHILLER',
        'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
        'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
        'ESTU_COMOCAPACITOEXAMENSB11',
        'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
        'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA']
    col_geo = 'ESTU_COD_DEPTO_PRESENTACION'

    for col in columnas_puntaje:
        if col in df_limpio.columns:
            df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce')
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mean())
    for col in columnas_ordinales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())
    for col in columnas_nominales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode(dropna=True)[0])

    cols_usar = columnas_ordinales + columnas_puntaje + columnas_nominales
    cols_extra = [col_geo, 'INST_COD_INSTITUCION', 'ESTU_PRGM_ACADEMICO',
                  'PERIODO', 'PUNT_GLOBAL', 'ESTU_CONSECUTIVO']
    cols_df = cols_usar + [c for c in cols_extra if c in df_limpio.columns]
    df_filtrado = df_limpio[[c for c in cols_df if c in df_limpio.columns]].copy()
    df_filtrado = df_filtrado.dropna(subset=[c for c in columnas_puntaje if c in df_filtrado.columns])
    print(f"Filas después de limpieza: {len(df_filtrado):,}")

    cols_punt = [c for c in columnas_puntaje if c in df_filtrado.columns]
    cols_ord = [c for c in columnas_ordinales if c in df_filtrado.columns]
    cols_nom = [c for c in columnas_nominales if c in df_filtrado.columns]

    scaler = StandardScaler()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_punt = scaler.fit_transform(df_filtrado[cols_punt])
    X_ohe = encoder.fit_transform(df_filtrado[cols_nom])
    X = np.hstack([df_filtrado[cols_ord].values, X_punt, X_ohe])
    feature_names = (cols_ord + list(scaler.get_feature_names_out(cols_punt))
                      + list(encoder.get_feature_names_out(cols_nom)))

    if np.isnan(X).any():
        from sklearn.impute import SimpleImputer
        X = SimpleImputer(strategy='median').fit_transform(X)
        print("NaN residuales imputados con mediana")

    if sample_n is not None and sample_n < df_filtrado.shape[0]:
        rng = np.random.default_rng(seed=random_state)
        idx = rng.choice(df_filtrado.shape[0], size=sample_n, replace=False)
        df_filtrado = df_filtrado.iloc[idx].reset_index(drop=True)
        X = X[idx]

    print(f"Preprocesamiento completo — shape X: {X.shape}")
    return df_limpio, df_filtrado, X, feature_names, encoder, scaler


In [ ]:
df_limpio_full, df_filtrado_full, X_full, feature_names, encoder, scaler = \
    preprocesar_saber_pro(df, sample_n=200_000)
n_total = X_full.shape[0]
print(f"Shape X_full: {X_full.shape}")

## 3. Ajustar UMAP en 2D-5D (se guarda en Drive; si la sesión se desconecta, esta celda reutiliza lo ya calculado)

⏱️ Cada dimensión toma varios minutos. Las 4 dimensiones (2D-5D) pueden tardar 20-30 minutos en total en Colab gratuito.

In [ ]:
def reducir_umap(X, n_components=2, random_state=42, **kwargs):
    reducer = umap_cpu.UMAP(n_components=n_components, random_state=random_state,
                             n_neighbors=10, low_memory=True, n_jobs=-1, **kwargs)
    if X.shape[0] > 100_000:
        rng = np.random.default_rng(42)
        idx_fit = rng.choice(X.shape[0], size=80_000, replace=False)
        reducer.fit(X[idx_fit])
        return reducer.transform(X)
    return reducer.fit_transform(X)

embeddings_path = os.path.join(OUT_DIR, "embeddings")
os.makedirs(embeddings_path, exist_ok=True)
embeddings = {}
for n_dim in [2, 3, 4, 5]:
    emb_file = os.path.join(embeddings_path, f"umap_{n_dim}d.npy")
    if os.path.exists(emb_file):
        embeddings[n_dim] = np.load(emb_file)
        print(f"[{n_dim}D] embedding cacheado, shape={embeddings[n_dim].shape}")
    else:
        t_d = time.time()
        print(f"[{n_dim}D] ajustando UMAP...")
        embeddings[n_dim] = reducir_umap(X_full, n_components=n_dim)
        np.save(emb_file, embeddings[n_dim])
        print(f"[{n_dim}D] listo en {time.time()-t_d:.1f}s")

## 4. Funciones de métricas (Silhouette, Calinski-Harabasz, Davies-Bouldin, Dunn)

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

def dunn_index(X_np, labels_np):
    unique = np.unique(labels_np)
    unique = unique[unique != -1]
    if len(unique) < 2:
        return np.nan
    clusters = [X_np[labels_np == l] for l in unique]
    centroids = [c.mean(axis=0) for c in clusters]
    min_inter = min(np.linalg.norm(centroids[i] - centroids[j])
                     for i in range(len(centroids)) for j in range(i + 1, len(centroids)))
    max_intra = max((np.linalg.norm(c - c.mean(axis=0), axis=1).max() * 2 if len(c) > 1 else 0)
                     for c in clusters) or 1
    return min_inter / max_intra

def calcular_metricas(X_np, labels_np, bic=None):
    mask = labels_np != -1
    X_eval, lbl_eval = X_np[mask], labels_np[mask]
    if len(set(lbl_eval)) < 2:
        return None
    sample = min(10_000, len(lbl_eval))
    sil = silhouette_score(X_eval, lbl_eval, sample_size=sample, random_state=42)
    return {'silhouette': round(float(sil), 4), 'calinski': round(float(calinski_harabasz_score(X_eval, lbl_eval)), 2),
            'davies_bouldin': round(float(davies_bouldin_score(X_eval, lbl_eval)), 4),
            'dunn': round(float(dunn_index(X_eval, lbl_eval)), 4),
            'bic': round(float(bic), 2) if bic is not None else None,
            'n_clusters': int(len(set(lbl_eval))), 'n_ruido': int((labels_np == -1).sum())}

## 5. A) DBSCAN en 2D-5D y B) HDBSCAN ampliado
⏱️ Esta celda es la más pesada del notebook (barre varios eps/min_samples por dimensión) — puede tardar 15-30 minutos.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import hdbscan

resultados = {"dbscan": [], "hdbscan": [], "gmm": []}

for n_dim, X_umap in embeddings.items():
    print(f"[{n_dim}D] DBSCAN: estimando eps via k-distance...")
    nn = NearestNeighbors(n_neighbors=5, n_jobs=-1).fit(X_umap)
    dists, _ = nn.kneighbors(X_umap)
    k_dists = np.sort(dists[:, -1])
    eps_vals = [round(np.percentile(k_dists, p), 3) for p in [10, 25, 50]]
    for eps in eps_vals:
        for min_s in [5, 10, 20]:
            try:
                labels = DBSCAN(eps=eps, min_samples=min_s, n_jobs=-1).fit_predict(X_umap)
                noise_pct = (labels == -1).mean() * 100
                row = {'algoritmo': 'DBSCAN', 'n_dim': n_dim, 'eps': eps,
                       'min_samples': min_s, 'noise_pct': round(noise_pct, 2)}
                m = calcular_metricas(X_umap, labels)
                if m: row.update(m)
                resultados["dbscan"].append(row)
            except Exception as e:
                print(f"[{n_dim}D] DBSCAN eps={eps} min_s={min_s} ERROR: {e}")
    print(f"[{n_dim}D] HDBSCAN...")
    for mc in [5, 10, 15, 50, 100, 250, 500, 1000]:
        for ms in [5, 10]:
            try:
                labels = hdbscan.HDBSCAN(min_cluster_size=mc, min_samples=ms).fit_predict(X_umap)
                noise_pct = (labels == -1).mean() * 100
                row = {'algoritmo': 'HDBSCAN', 'n_dim': n_dim, 'min_cluster_size': mc,
                       'min_samples': ms, 'noise_pct': round(noise_pct, 2)}
                m = calcular_metricas(X_umap, labels)
                if m: row.update(m)
                resultados["hdbscan"].append(row)
            except Exception as e:
                print(f"[{n_dim}D] HDBSCAN mc={mc} ms={ms} ERROR: {e}")
    json.dump(resultados, open(os.path.join(OUT_DIR, "resultados.json"), "w"), indent=2)
    print(f"[{n_dim}D] listo — {len(resultados['dbscan'])} filas DBSCAN, "
          f"{len(resultados['hdbscan'])} filas HDBSCAN acumuladas")

## 6. C) Sensibilidad de GMM (covariance_type × n_init, en K=2 y K=8)

In [ ]:
from sklearn.mixture import GaussianMixture

X_2d = embeddings[2]
for k in [2, 8]:
    for cov in ['full', 'tied', 'diag', 'spherical']:
        for n_init in [1, 10]:
            try:
                gmm = GaussianMixture(n_components=k, random_state=42,
                                       covariance_type=cov, n_init=n_init)
                labels = gmm.fit_predict(X_2d)
                bic = gmm.bic(X_2d)
                row = {'algoritmo': 'GMM', 'n_dim': 2, 'k': k,
                       'covariance_type': cov, 'n_init': n_init}
                m = calcular_metricas(X_2d, labels, bic=bic)
                if m: row.update(m)
                resultados["gmm"].append(row)
                print(f"GMM k={k} cov={cov} n_init={n_init} -> "
                      f"sil={m['silhouette'] if m else None} bic={bic:.1f}")
            except Exception as e:
                print(f"GMM k={k} cov={cov} n_init={n_init} ERROR: {e}")

json.dump(resultados, open(os.path.join(OUT_DIR, "resultados.json"), "w"), indent=2)
print("LISTO.")

## 7. Resumen: ¿converge HDBSCAN a 8 clústeres con min_cluster_size grande?

In [ ]:
import pandas as pd
hdb_df = pd.DataFrame(resultados["hdbscan"])
print(hdb_df[hdb_df['min_cluster_size'].isin([250, 500])]
      [['n_dim', 'min_cluster_size', 'min_samples', 'n_clusters', 'noise_pct']]
      .to_string(index=False))
print("\nValor esperado (manuscrito): ~8 clústeres, ruido bajo, en mc=250-500 "
      "consistentemente en 2D-5D.")

## 8. Figura (Supplementary Figure S4)
Reproduce la Figura S4 de los Materiales Suplementarios a partir de los resultados ya calculados arriba (`resultados["hdbscan"]` y `resultados["dbscan"]`).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

hdb_df = pd.DataFrame(resultados["hdbscan"])
dbs_df = pd.DataFrame(resultados["dbscan"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)

colors = {2: '#3B6FA0', 3: '#4C9F70', 4: '#B33F3F', 5: '#8B6FB3'}
for n_dim in [2, 3, 4, 5]:
    sub = hdb_df[(hdb_df['n_dim'] == n_dim) & (hdb_df['min_samples'] == 5)].sort_values('min_cluster_size')
    axes[0].plot(sub['min_cluster_size'], sub['n_clusters'], marker='o', label=f'{n_dim}D', color=colors[n_dim])
axes[0].set_xscale('log')
axes[0].axhline(8, color='gray', linestyle='--', linewidth=1)
axes[0].set_xlabel('min_cluster_size (escala log)')
axes[0].set_ylabel('N.\u00b0 de cl\u00fasteres encontrados (HDBSCAN)')
axes[0].set_title('HDBSCAN converge a ~8 cl\u00fasteres\ncon min_cluster_size realista (250-500)')
axes[0].legend(title='Dim. UMAP')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

data_box = [dbs_df[dbs_df['n_dim'] == n_dim]['noise_pct'].values for n_dim in [2, 3, 4, 5]]
bp = axes[1].boxplot(data_box, tick_labels=['2D', '3D', '4D', '5D'], patch_artist=True, widths=0.5)
for patch in bp['boxes']:
    patch.set_facecolor('#C0605F')
    patch.set_alpha(0.75)
axes[1].axhline(20, color='gray', linestyle='--', linewidth=1, label='20% ruido (referencia)')
axes[1].set_ylabel('% de puntos etiquetados como ruido')
axes[1].set_title('DBSCAN: alto % de ruido en TODAS\nlas dimensiones probadas (2D-5D)')
axes[1].legend(loc='lower left', fontsize=8)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figura_comparacion_algoritmos.png'), dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("\u2705 Figura guardada en Drive (Supplementary Figure S4)")
